# OCR Comparison: EasyOCR vs PaddleOCR

Сравнение результатов двух OCR-движков на одних и тех же мемах при разных уровнях confidence.

In [ ]:
import csv
import random
import textwrap
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
from IPython.display import display, HTML

ROOT = Path('.').resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
%matplotlib inline
plt.rcParams["figure.dpi"] = 120

## Загрузка данных

In [ ]:
easy = {}
with open(ROOT / "data/processed/metadata_ocr.csv") as f:
    for row in csv.DictReader(f):
        easy[row["filename"]] = row

paddle = {}
for pf in (ROOT / "data").rglob("ocr_paddle.csv"):
    with open(pf) as f:
        for row in csv.DictReader(f):
            fn = row.get("filename", "")
            paddle[fn] = dict(row)
            paddle[fn]["source_path"] = str(pf.parent / fn)

common = set(easy.keys()) & set(paddle.keys())
print(f"EasyOCR entries:  {len(easy)}")
print(f"PaddleOCR entries: {len(paddle)}")
print(f"Common (both):     {len(common)}")

## Распределение confidence

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

easy_confs = [float(easy[fn].get("confidence", 0)) for fn in common if float(easy[fn].get("confidence", 0)) > 0]
paddle_confs = [float(paddle[fn].get("confidence", 0)) for fn in common if float(paddle[fn].get("confidence", 0)) > 0]

axes[0].hist(easy_confs, bins=50, color="steelblue", alpha=0.8, edgecolor="white")
axes[0].axvline(x=0.6, color="red", linestyle="--", label="threshold=0.6")
axes[0].set_title("EasyOCR Confidence Distribution")
axes[0].set_xlabel("Confidence")
axes[0].legend()

axes[1].hist(paddle_confs, bins=50, color="coral", alpha=0.8, edgecolor="white")
axes[1].axvline(x=0.6, color="red", linestyle="--", label="threshold=0.6")
axes[1].set_title("PaddleOCR Confidence Distribution")
axes[1].set_xlabel("Confidence")
axes[1].legend()

plt.tight_layout()
plt.show()

## Примеры OCR при разных уровнях confidence

In [ ]:
targets = [
    (0.6,  0.08, "Low (~0.6)"),
    (0.75, 0.08, "Medium (~0.75)"),
    (0.92, 0.08, "High (~0.92)"),
]

random.seed(42)
selected = []
for target, delta, label in targets:
    candidates = []
    for fn in sorted(common):
        ec = float(easy[fn].get("confidence", 0))
        pc = float(paddle[fn].get("confidence", 0))
        et = easy[fn].get("ocr_text", "").strip()
        pt = paddle[fn].get("ocr_text", "").strip()
        sp = easy[fn].get("source_path", "")
        if not sp:
            sp = paddle[fn].get("source_path", "")
        if abs(ec - target) < delta and ec > 0 and et and pt and Path(sp).exists():
            candidates.append((fn, ec, pc, et, pt, sp, label))
    random.shuffle(candidates)
    selected.extend(candidates[:3])
print(f"Selected {len(selected)} examples")

In [ ]:
def show_ocr_example(fn, ec, pc, et, pt, sp, label):
    fig, axes = plt.subplots(1, 2, figsize=(16, 5), gridspec_kw={"width_ratios": [1, 1.6]})
    
    try:
        img = Image.open(sp).convert("RGB")
        img.thumbnail((500, 500))
        axes[0].imshow(np.array(img))
    except Exception:
        axes[0].text(0.5, 0.5, "Image not found", ha="center", va="center", fontsize=14)
    axes[0].set_title(f"{label}
{fn}", fontsize=11, fontweight="bold")
    axes[0].axis("off")
    
    et_w = textwrap.fill(et[:200], width=55)
    pt_w = textwrap.fill(pt[:200], width=55)
    text_block = f"EasyOCR  [conf={ec:.3f}]:
{et_w}

PaddleOCR [conf={pc:.3f}]:
{pt_w}"
    axes[1].text(0.05, 0.95, text_block, transform=axes[1].transAxes,
                 fontsize=10, fontfamily="monospace", verticalalignment="top",
                 bbox=dict(boxstyle="round,pad=0.5", facecolor="lightyellow", alpha=0.8))
    
    easy_color = "green" if ec >= 0.8 else ("orange" if ec >= 0.6 else "red")
    paddle_color = "green" if pc >= 0.8 else ("orange" if pc >= 0.6 else "red")
    axes[1].barh([1, 0], [ec, pc], color=[easy_color, paddle_color], height=0.3, alpha=0.3)
    axes[1].set_xlim(0, 1.0)
    axes[1].set_yticks([1, 0])
    axes[1].set_yticklabels(["EasyOCR", "PaddleOCR"], fontsize=10)
    axes[1].set_xlabel("Confidence")
    axes[1].axvline(x=0.6, color="red", linestyle="--", alpha=0.5, label="threshold=0.6")
    axes[1].legend(fontsize=8)
    plt.tight_layout()
    plt.show()

### Low Confidence (~0.6)

Тексты с низкой уверенностью — заметный шум, ошибки в символах.

In [ ]:
for item in selected[:3]:
    show_ocr_example(*item)

### Medium Confidence (~0.75)

Текст читаем, но отдельные ошибки присутствуют.

In [ ]:
for item in selected[3:6]:
    show_ocr_example(*item)

### High Confidence (~0.92)

Оба движка работают хорошо.

In [ ]:
for item in selected[6:9]:
    show_ocr_example(*item)

## Выводы

| Confidence | Качество | Действие |
|-----------|----------|----------|
| < 0.6 | Шум, ошибки в символах | Отфильтровано (порог 0.6) |
| 0.6–0.8 | Текст читаем, отдельные артефакты | Сохранено |
| > 0.8 | Хорошее качество | Сохранено |

**Наблюдения:**
- PaddleOCR в среднем выдаёт более высокий confidence при том же качестве
- На русском тексте PaddleOCR заметно точнее
- Порог 0.6 выбран эмпирически (визуальная проверка ~150 примеров), формальная валидация через gold-набор запланирована
